In [1]:
import subprocess, sys, os

# 1. Install high-performance libraries
packages = ["faster-whisper", "huggingface_hub", "hf-transfer", "pandas", "torch"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)

# 2. Install FFmpeg with sudo to avoid permission locks
subprocess.run(["sudo", "apt-get", "update", "-y", "-q"], check=True)
subprocess.run(["sudo", "apt-get", "install", "-y", "-q", "ffmpeg"], check=True)

# Enable Rust-based fast downloads
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("✅ Environment ready")

Get:1 https://us-east-2.ec2.archive.ubuntu.com/ubuntu noble InRelease [256 kB]
Hit:2 https://nvidia.github.io/libnvidia-container/stable/deb/amd64  InRelease
Get:3 https://download.docker.com/linux/ubuntu noble InRelease [48.5 kB]
Get:4 https://us-east-2.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:5 https://us-east-2.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:6 https://us-east-2.ec2.archive.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:7 https://cli.github.com/packages stable InRelease [3917 B]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1581 B]
Get:9 https://download.docker.com/linux/ubuntu noble/stable amd64 Packages [59.7 kB]
Get:10 https://us-east-2.ec2.archive.ubuntu.com/ubuntu noble/main amd64 Packages [1808 kB]
Hit:11 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease
Get:12 https://us-east-2.ec2.archive.ubuntu.com/ubuntu noble amd64 Contents (deb) [51.3 MB]
G

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.



Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 36 not upgraded.
✅ Environment ready


In [ ]:
import torch
from faster_whisper import WhisperModel, BatchedInferencePipeline

# --- CONFIG ---
MODEL_ID        = "large-v3"
DATASET_REPO_ID = "frovolts/Lipi-Ghor-bn-882-SSTT"
OUTPUT_REPO_ID  = "hasans090/whisperv3_inference_lp"
HF_TOKEN        = "______________" # Replace with your token

VERSION           = 5
FILES_PER_VERSION = 205
OUTPUT_CSV        = f"whisperv3_lipighor_v{VERSION}.csv"
LOCAL_AUDIO_CACHE = "/tmp/audio_cache"
os.makedirs(LOCAL_AUDIO_CACHE, exist_ok=True)

def make_pipeline(gpu_id):
    print(f"Loading model on GPU {gpu_id} with int8_float16...")
    model = WhisperModel(
        MODEL_ID,
        device="cuda",
        device_index=gpu_id,
        compute_type="int8_float16", # Sweet spot for speed on Ada
        cpu_threads=36                # Faster VAD and audio decoding
    )
    return BatchedInferencePipeline(model=model)

# Dynamic GPU detection
num_gpus = torch.cuda.device_count()
pipes = [make_pipeline(i) for i in range(num_gpus)]
print(f"✅ {len(pipes)} Pipeline(s) ready.")

Loading model on GPU 0 with int8_float16...


✅ 1 Pipeline(s) ready.


In [13]:
import pandas as pd
from huggingface_hub import hf_hub_download, list_repo_files, upload_file
from pathlib import Path

def get_status():
    # 1. Get total target files for this version
    all_files = list_repo_files(DATASET_REPO_ID, repo_type="dataset", token=HF_TOKEN)
    audio_files = sorted([f for f in all_files if f.endswith(('.wav', '.mp3', '.flac'))])
    start, end = (VERSION-1)*FILES_PER_VERSION, VERSION*FILES_PER_VERSION
    targets = audio_files[start:end]
    
    # 2. Check what is already done on HF
    try:
        path = hf_hub_download(repo_id=OUTPUT_REPO_ID, filename=OUTPUT_CSV, 
                               repo_type="dataset", token=HF_TOKEN, force_download=True)
        df = pd.read_csv(path)
        done_ids = set(df["id"].astype(str).tolist())
        return targets, done_ids, df.to_dict("records")
    except:
        return targets, set(), []

target_files, done_ids, results = get_status()
remaining = [f for f in target_files if Path(f).stem not in done_ids]

print(f"📊 Total in version: {len(target_files)} | Done: {len(done_ids)} | Remaining: {len(remaining)}")

whisperv3_lipighor_v5.csv: 0.00B [00:00, ?B/s]

📊 Total in version: 205 | Done: 60 | Remaining: 145


In [ ]:
import threading
from queue import Queue

download_queue = Queue(maxsize=3) # Prefetch 3 files ahead

def download_worker():
    for f_path in remaining:
        try:
            p = hf_hub_download(repo_id=DATASET_REPO_ID, filename=f_path, 
                                repo_type="dataset", token=HF_TOKEN, local_dir=LOCAL_AUDIO_CACHE)
            download_queue.put((Path(f_path).stem, p))
        except Exception as e:
            print(f"Download error: {e}")
    download_queue.put(None)

# Start background downloader
threading.Thread(target=download_worker, daemon=True).start()

print("🚀 Processing started with Background Prefetching...")

while True:
    item = download_queue.get()
    if item is None: break
    
    file_id, local_path = item
    
    try:
        # Optimized for speed: Greedy decoding (beam_size=1)
        segments, _ = pipes[0].transcribe(
            str(local_path),
            language="bn",
            beam_size=1,   
            batch_size=64, # High throughput for RTX 6000
            vad_filter=True
        )
        transcript = " ".join(seg.text for seg in segments).strip()
        results.append({"id": file_id, "transcript": transcript})
    except Exception as e:
        print(f"❌ Transcription error on {file_id}: {e}")
    
    if os.path.exists(local_path): os.remove(local_path)
    
    # Periodic Save & Push
    if len(results) % 5 == 0:
        pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
        upload_file(path_or_fileobj=OUTPUT_CSV, path_in_repo=OUTPUT_CSV, 
                    repo_id=OUTPUT_REPO_ID, repo_type="dataset", token=HF_TOKEN)
        print(f"📤 Progress: {len(results)}/{len(target_files)} uploaded.")

print("✅ Batch complete.")

🚀 Processing started with Background Prefetching...


data/rfhpQBQY8RY.mp3:   0%|          | 0.00/24.3M [00:00<?, ?B/s]

data/rh7fqyOJrlw.mp3:   0%|          | 0.00/73.4M [00:00<?, ?B/s]

data/ri5T-1B6DpA.mp3:   0%|          | 0.00/7.76M [00:00<?, ?B/s]

data/rk1DVLgn9z8.mp3:   0%|          | 0.00/4.21M [00:00<?, ?B/s]

...

data/xdJ3FQXTZ4g.mp3:   0%|          | 0.00/46.0M [00:00<?, ?B/s]

📤 Progress: 155/205 uploaded.


data/xegGMB6lsc4.mp3:   0%|          | 0.00/23.6M [00:00<?, ?B/s]

data/xje5hypGYWU.mp3:   0%|          | 0.00/4.54M [00:00<?, ?B/s]

data/xjzesk9koE0.mp3:   0%|          | 0.00/44.0M [00:00<?, ?B/s]